# PokerBench Oracle Baseline

PokerBench est maintenant la baseline oracle principale pour entra?ner CHECK / FOLD / CALL / RAISE avec des labels solver s?par?s des labels live/legacy.

## Param?tres

Le notebook r?utilise les artefacts existants par d?faut. Mets `FORCE_REBUILD = True` seulement si tu veux relancer l'entra?nement.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, SVG, display

ROOT = Path.cwd()
DATA_DIR = ROOT / "data/pokerbench"
MODEL_DIR = ROOT / "outputs/readiness/pokerbench_oracle_baseline_v1"
REPORT_PATH = MODEL_DIR / "training_report.json"
CONTRACT_PATH = MODEL_DIR / "feature_contract.json"
CANDIDATES_PATH = MODEL_DIR / "candidates.csv"
COMPARISON_PATH = MODEL_DIR / "comparison_with_live_bb_baseline_v1.md"
GRAPHICAL_STUDY_PATH = MODEL_DIR / "graphical_study.md"

RUN_TRAINING = True
RUN_GRAPHICAL_STUDY = True
FAST_MODE = True
FORCE_REBUILD = False
MAX_ROWS = None  # Exemple: 12000 pour un test rapide; None utilise les CSV structur?s disponibles.


def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def run_cmd(args):
    started = time.perf_counter()
    completed = subprocess.run(args, cwd=ROOT, text=True, capture_output=True)
    elapsed = time.perf_counter() - started
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    completed.check_returncode()
    print(f"temps_execution_secondes={elapsed:.2f}")
    return elapsed

## 1. Pr?parer / entra?ner

In [ ]:
missing_required = not REPORT_PATH.exists() or not CANDIDATES_PATH.exists() or not (MODEL_DIR / "model.joblib").exists()
missing_graphs = RUN_GRAPHICAL_STUDY and not all((MODEL_DIR / name).exists() for name in [
    "eda_label_distribution.svg",
    "eda_street_distribution.svg",
    "eda_label_by_street.svg",
    "confusion_matrix.svg",
    "feature_importance.svg",
    "feature_correlation.svg",
    "learning_curve.svg",
    "graphical_study.md",
])

if RUN_TRAINING and (FORCE_REBUILD or missing_required or missing_graphs):
    cmd = [
        sys.executable,
        "experiments/pokerbench_oracle_baseline_v1.py",
        "--data-dir",
        str(DATA_DIR),
        "--output-dir",
        str(MODEL_DIR),
    ]
    if DATA_DIR.exists():
        cmd.append("--no-download")
    if MAX_ROWS is not None:
        cmd += ["--max-rows", str(MAX_ROWS)]
    if not FAST_MODE:
        cmd.append("--full-diagnostics")
    run_cmd(cmd)
else:
    print("Artefacts PokerBench r?utilis?s depuis le cache local.")

## 2. R?sum? du mod?le

In [ ]:
report = load_json(REPORT_PATH)
summary = {
    "baseline": "PokerBench oracle",
    "rows_loaded": report["rows_loaded"],
    "rows_usable": report["rows_usable"],
    "model_type": report["model_type"],
    "accuracy": report["accuracy"],
    "macro_f1": report["macro_f1"],
    "offline_prediction": report["offline_prediction"]["status"],
    "bot_live_connection": report["bot_live_connection"],
}
display(pd.DataFrame([summary]))
display(pd.Series(report["label_distribution"], name="count").to_frame())
display(pd.Series(report["preflop_postflop_distribution"], name="count").to_frame())

## 3. Contrat features

In [ ]:
contract = load_json(CONTRACT_PATH)
print("classes:", contract["allowed_predictions"])
print("label_source:", contract["label_source"])
print("CALL natif:", contract["call_is_native_class"])
print("leakage utilis? par le mod?le:", contract["leakage_columns_used_by_model"])
display(pd.DataFrame({"feature": contract["features_model_used"]}))

## 4. Distributions

In [ ]:
for name in ["eda_label_distribution.svg", "eda_street_distribution.svg", "eda_label_by_street.svg"]:
    path = MODEL_DIR / name
    if path.exists():
        display(SVG(filename=str(path)))

## 5. Performance globale et par classe

In [ ]:
if (MODEL_DIR / "confusion_matrix.svg").exists():
    display(SVG(filename=str(MODEL_DIR / "confusion_matrix.svg")))

rows = []
for label in ["CHECK", "FOLD", "CALL", "RAISE"]:
    metrics = report["classification_report"].get(label, {})
    rows.append({
        "label": label,
        "precision": metrics.get("precision"),
        "recall": metrics.get("recall"),
        "f1": metrics.get("f1-score"),
        "support": metrics.get("support"),
    })
display(pd.DataFrame(rows))

## 6. Performance par street

In [ ]:
perf = report.get("performance_by_street", {})
display(pd.DataFrame.from_dict(perf, orient="index").sort_index())

## 7. Features et corr?lations

In [ ]:
for name in ["feature_importance.svg", "feature_correlation.svg"]:
    path = MODEL_DIR / name
    if path.exists():
        display(SVG(filename=str(path)))

## 8. Learning curve

In [ ]:
path = MODEL_DIR / "learning_curve.svg"
if path.exists():
    display(SVG(filename=str(path)))

## 9. Synth?se graphique

In [ ]:
if GRAPHICAL_STUDY_PATH.exists():
    display(Markdown(GRAPHICAL_STUDY_PATH.read_text(encoding="utf-8")))

## 10. Comparaison baseline historique

In [ ]:
if COMPARISON_PATH.exists():
    display(Markdown(COMPARISON_PATH.read_text(encoding="utf-8")))

## 11. Pr?diction offline

In [ ]:
from experiments.pokerbench_oracle_baseline_v1 import predict_first_row

sample = pd.read_csv(CANDIDATES_PATH, nrows=1).iloc[0].to_dict()
predict_first_row(MODEL_DIR, sample)

## 12. Archives / baselines historiques

Les anciens flux restent dans `outputs/readiness/` pour comparaison, mais ils ne sont plus le chemin principal de ce notebook.

In [ ]:
historical = []
for path in [ROOT / "outputs/readiness/live_bb_baseline_v1/training_report.json"]:
    if path.exists():
        item = load_json(path)
        historical.append({
            "name": path.parent.name,
            "rows": item.get("rows_used"),
            "label_source": item.get("label_source") or item.get("generation_mode"),
            "accuracy": item.get("accuracy"),
            "macro_f1": item.get("macro_f1"),
        })
display(pd.DataFrame(historical))

## 13. Tests

In [ ]:
RUN_TESTS = False

if RUN_TESTS:
    run_cmd([sys.executable, "-m", "pytest"])
else:
    print("Tests saut?s: RUN_TESTS = False")